# 7. Live Data Streaming App

**Python for Data Engineering** — Module 7

---

## What you will learn

| Concept | Description |
|---------|-------------|
| Producer / Consumer pattern | How real-time data flows between components |
| `queue.Queue` | Thread-safe in-memory message broker |
| Apache Kafka | Distributed streaming platform |
| Rolling analytics | SMA, VWAP, volatility computed on a sliding window |
| Streamlit | Building a live dashboard |
| Docker Compose | Spinning up Kafka locally |

---

## Architecture overview

```
┌──────────────────────────────────────────────────┐
│              Live Streaming App                  │
│                                                  │
│  ┌────────────┐   ticks    ┌──────────────────┐  │
│  │  Producer  │ ─────────► │    Message Bus   │  │
│  │ (generator)│            │  Queue / Kafka   │  │
│  └────────────┘            └────────┬─────────┘  │
│                                     │ ticks      │
│                                     ▼            │
│                            ┌────────────────┐    │
│                            │    Consumer    │    │
│                            │  + Analytics  │    │
│                            └───────┬────────┘    │
│                                    │ enriched    │
│                                    ▼            │
│                          ┌──────────────────┐   │
│                          │  Streamlit       │   │
│                          │  Live Dashboard  │   │
│                          └──────────────────┘   │
└──────────────────────────────────────────────────┘
```

**Three transport modes:**
- **In-memory** (default): `queue.Queue` — no setup required, perfect for development.
- **Kafka** (optional): Set `USE_KAFKA=true` in `.env` and start the Docker stack.


---
## 1. Setup

Install the required packages:

In [ ]:
# Install dependencies (run once)
# !pip install streamlit plotly pandas numpy kafka-python python-dotenv

In [ ]:
import sys
import os

# Make sure the module directory is on the path when running from a notebook
module_dir = os.path.dirname(os.path.abspath('__file__')) if '__file__' in dir() else os.getcwd()
if module_dir not in sys.path:
    sys.path.insert(0, module_dir)

import queue
import time
import json
import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots

print('Imports OK')

---
## 2. Data Generator

The generator produces realistic market-data ticks using a **Geometric Brownian Motion** (GBM) model — the same model used in the famous Black-Scholes options pricing formula.

### The price model

$$P_{t+1} = P_t \cdot e^{(\mu + \sigma \cdot Z)}$$

- $P_t$ — current price  
- $\mu$ — drift (mean-reversion pull toward starting price)  
- $\sigma$ — per-tick volatility  
- $Z \sim \mathcal{N}(0,1)$ — random shock

In [ ]:
from data_generator import MarketDataGenerator, SensorDataGenerator

# --- Market data ---
gen = MarketDataGenerator(symbols=['AAPL', 'MSFT', 'BTC-USD'])

# Generate 5 ticks for AAPL
for _ in range(5):
    tick = gen.generate_tick('AAPL')
    print(json.dumps(tick, indent=2))
    print()

In [ ]:
# Generate a batch and inspect as a DataFrame
gen2 = MarketDataGenerator(symbols=['AAPL', 'GOOGL', 'TSLA'])
batch = gen2.generate_batch(n=150)
df_batch = pd.DataFrame(batch)
print(df_batch.shape)
df_batch.head(10)

In [ ]:
# Quick price-path visualisation
fig = go.Figure()
for sym, grp in df_batch.groupby('symbol'):
    fig.add_trace(go.Scatter(x=grp.index, y=grp['price'], mode='lines', name=sym))

fig.update_layout(
    title='Simulated price paths (150 ticks)',
    xaxis_title='Tick',
    yaxis_title='Price (USD)',
    template='plotly_dark',
    height=400,
)
fig.show()

In [ ]:
# IoT sensor data
sensor_gen = SensorDataGenerator()
readings = [sensor_gen.generate_reading() for _ in range(50)]
df_sensors = pd.DataFrame(readings)
print(df_sensors.dtypes)
df_sensors.head()

---
## 3. Producer — sending data to the stream

The producer runs in a **background daemon thread** so the notebook stays interactive.

```
Generator → Producer → Queue
```

In [ ]:
from producer import InMemoryProducer

# Shared queue — this is the 'message bus'
raw_queue = queue.Queue(maxsize=1000)

# Create and start the producer
producer = InMemoryProducer(
    data_queue=raw_queue,
    symbols=['AAPL', 'GOOGL', 'MSFT', 'TSLA'],
    tick_interval=0.2,    # 0.2 s per round → ~20 ticks/sec
)
producer.start()

print('Producer running:', producer.is_running)

# Wait a moment then check queue size
time.sleep(1)
print('Items in queue after 1s:', raw_queue.qsize())

In [ ]:
# Peek at 3 raw ticks
for _ in range(3):
    tick = raw_queue.get(timeout=2)
    print(json.dumps(tick, indent=2))

---
## 4. Consumer — processing the stream

The consumer reads ticks from the queue, computes rolling analytics, and pushes enriched records downstream.

### Rolling analytics computed per tick

| Metric | Formula | Window |
|--------|---------|--------|
| **SMA-20** | $\frac{1}{20}\sum_{i=t-19}^{t} P_i$ | Last 20 ticks |
| **SMA-50** | $\frac{1}{50}\sum_{i=t-49}^{t} P_i$ | Last 50 ticks |
| **VWAP** | $\frac{\sum P_i V_i}{\sum V_i}$ | Rolling window |
| **Volatility** | $\sigma(\ln P_i / P_{i-1})$ | Last 20 log-returns |

In [ ]:
from consumer import InMemoryConsumer

processed_queue = queue.Queue(maxsize=1000)

consumer = InMemoryConsumer(
    input_queue=raw_queue,
    processed_queue=processed_queue,
    export_path='stream_output.jsonl',   # optional: write to file
)
consumer.start()

print('Consumer running:', consumer.is_running)

In [ ]:
# Collect 80 enriched ticks
time.sleep(3)

enriched_ticks = []
while not processed_queue.empty():
    enriched_ticks.append(processed_queue.get_nowait())

df_enriched = pd.DataFrame(enriched_ticks)
print(f'Collected {len(df_enriched)} enriched ticks')
df_enriched[['symbol','price','sma_20','sma_50','vwap','volatility','alert']].tail(10)

In [ ]:
# Summary statistics per symbol
consumer.analytics.summary_dataframe()

---
## 5. Visualising the live stream

We plot the enriched stream with SMA overlays and volume bars.

In [ ]:
# Collect more data for a nicer chart
time.sleep(5)
while not processed_queue.empty():
    enriched_ticks.append(processed_queue.get_nowait())

df_all = pd.DataFrame(enriched_ticks)
df_all['timestamp'] = pd.to_datetime(df_all['timestamp'])
print(f'Total ticks: {len(df_all)}')

symbol_to_plot = 'AAPL'
df_sym = df_all[df_all['symbol'] == symbol_to_plot].reset_index(drop=True)
print(f'{symbol_to_plot}: {len(df_sym)} ticks')

In [ ]:
fig = make_subplots(
    rows=2, cols=1,
    shared_xaxes=True,
    vertical_spacing=0.05,
    row_heights=[0.7, 0.3],
    subplot_titles=[f'{symbol_to_plot} Price Stream', 'Volume'],
)

# Price line
fig.add_trace(go.Scatter(
    x=df_sym['timestamp'], y=df_sym['price'],
    mode='lines', name='Price',
    line=dict(color='#00b4d8', width=1.5)
), row=1, col=1)

# SMA-20
sma20 = df_sym['sma_20'].dropna()
fig.add_trace(go.Scatter(
    x=df_sym.loc[sma20.index, 'timestamp'], y=sma20,
    mode='lines', name='SMA-20',
    line=dict(color='#f77f00', width=1, dash='dot')
), row=1, col=1)

# VWAP
vwap = df_sym['vwap'].dropna()
fig.add_trace(go.Scatter(
    x=df_sym.loc[vwap.index, 'timestamp'], y=vwap,
    mode='lines', name='VWAP',
    line=dict(color='#80b918', width=1, dash='longdash')
), row=1, col=1)

# Volume
colors = ['#2dc653' if df_sym['price'].iloc[i] >= df_sym['price'].iloc[max(0, i-1)] else '#e63946'
          for i in range(len(df_sym))]
fig.add_trace(go.Bar(
    x=df_sym['timestamp'], y=df_sym['volume'],
    marker_color=colors, name='Volume', showlegend=False
), row=2, col=1)

fig.update_layout(
    height=550, template='plotly_dark',
    margin=dict(l=10, r=10, t=40, b=10),
    hovermode='x unified'
)
fig.show()

---
## 6. Alerts

Alerts fire when a price deviates from its starting value by more than a threshold.

In [ ]:
alerts = consumer.analytics.alerts
if alerts:
    df_alerts = pd.DataFrame(alerts)
    print(f'{len(df_alerts)} alert(s) triggered:')
    display(df_alerts)
else:
    print('No alerts yet — run longer or lower ALERT_THRESHOLD_HIGH/LOW in config.py')

---
## 7. Kafka (optional)

For production-grade streaming, swap the in-memory queue for Kafka.

### Step 1 — start the Docker stack

```bash
docker compose up -d
```

This starts:
- **Zookeeper** (port 2181)
- **Kafka broker** (port 9092)
- **Kafka UI** (port 8080 → http://localhost:8080)

### Step 2 — configure

```bash
cp .env.example .env
# Edit .env and set USE_KAFKA=true
```

### Step 3 — run producer & consumer

```bash
# Terminal 1
python producer.py

# Terminal 2
python consumer.py
```

### Kafka vs in-memory queue

| Feature | `queue.Queue` | Apache Kafka |
|---------|--------------|-------------|
| Setup | None | Docker / cloud |
| Persistence | No | Yes (configurable retention) |
| Multiple consumers | 1 thread | Many consumer groups |
| Replay | No | Yes (seek to offset) |
| Scale | Single process | Distributed cluster |
| Best for | Dev / testing | Production |


In [ ]:
# Kafka producer example (requires USE_KAFKA=true and running broker)

# from producer import KafkaStreamProducer
# kafka_producer = KafkaStreamProducer(symbols=['AAPL', 'GOOGL'])
# kafka_producer.start()
# # ... do work ...
# kafka_producer.stop()

print('Uncomment the lines above after starting the Docker stack.')

---
## 8. Live Streamlit Dashboard

Launch the full interactive dashboard from a terminal:

```bash
streamlit run streaming_dashboard.py
```

Then open **http://localhost:8501** in your browser.

### Dashboard features

| Feature | Description |
|---------|-------------|
| Ticker cards | Live price + % change for each symbol |
| Line / Candlestick charts | Toggle via sidebar |
| SMA-20 / SMA-50 overlays | Enable in sidebar |
| VWAP | Always shown |
| Volume bars | Green = up-tick, red = down-tick |
| Alert log | Shows PRICE_SURGE / PRICE_DROP events |
| CSV export | Download snapshot of all ticks |
| Auto-refresh | Configurable interval (500 ms – 5 s) |

---
## 9. Extending the app

### Ideas for further development

1. **Multiple topics** — add a separate Kafka topic for alerts and consume it with a dedicated alert-handler service.
2. **Persistent storage** — write enriched ticks to PostgreSQL, DuckDB, or S3 Parquet for historical analysis.
3. **Machine-learning signals** — train an anomaly-detection model on the rolling features and emit predictions as a new stream.
4. **Spark Structured Streaming** — replace the Python consumer with a PySpark job for horizontal scale.
5. **Cloud deployment** — replace Docker Kafka with Amazon MSK, Confluent Cloud, or Azure Event Hubs (all Kafka-compatible).
6. **Real data** — swap `data_generator.py` for a live WebSocket feed (e.g., Alpaca, Binance, CoinGecko).

In [ ]:
# Cleanup — stop producer and consumer when done with the notebook
producer.stop()
consumer.stop()
print('Producer stopped:', not producer.is_running)
print('Consumer stopped:', not consumer.is_running)